In [3]:
# SIMPLE & EFFICIENT GDELT NEWS SCRAPER
# No batching complexity - just works!

import pandas as pd
import requests
import time
from datetime import datetime
import os
import json
from urllib.parse import urlparse

# ============================================================================
# SIMPLE CONFIGURATION
# ============================================================================

# Your assets
ASSETS = {
    'NVDA': 'NVIDIA Corporation',
    'MSFT': 'Microsoft Corporation', 
    'GOOGL': 'Alphabet Inc',
    'AMD': 'Advanced Micro Devices',
    'MU': 'Micron Technology Inc',
    'MRVL': 'Marvell Technology Group',
    'ASML': 'ASML Holding NV',
    'AEM': 'Agnico Eagle Mines Ltd',
    'VERU': 'Veru Inc',
    'AI': 'C3.ai Inc',
    'INGM': 'Inogen Inc',
    'PLUG': 'Plug Power Inc',
    'IONQ': 'IonQ Inc',
    'RGTI': 'Rigetti Computing Inc',
    'ARBE': 'Arbe Robotics Ltd',
    'RDDT': 'Reddit Inc',
    'SMR': 'NuScale Power Corporation'
}

# Quality news sources
QUALITY_SOURCES = {
    "reuters.com", "bloomberg.com", "wsj.com", "ft.com", "cnbc.com",
    "marketwatch.com", "barrons.com", "forbes.com", "yahoo.com"
}

# Simple settings
API_URL = "https://api.gdeltproject.org/api/v2/doc/doc"
SLEEP_TIME = 5  # 5 seconds between requests
MAX_ARTICLES = 250  # Max per request

# Create directories
os.makedirs("data/raw/news", exist_ok=True)

print("🚀 Simple GDELT News Scraper")
print(f"📊 Assets: {len(ASSETS)}")
print(f"⏱️ Sleep time: {SLEEP_TIME} seconds")

# ============================================================================
# SIMPLE FUNCTIONS
# ============================================================================

def test_gdelt():
    """Test if GDELT works"""
    print("🧪 Testing GDELT...")
    
    params = {
        "query": "Microsoft",
        "mode": "ArtList", 
        "format": "json",
        "maxrecords": "5"
    }
    
    try:
        response = requests.get(API_URL, params=params, timeout=10)
        if response.status_code == 200:
            articles = response.json().get("articles", [])
            print(f"✅ GDELT working! Got {len(articles)} test articles")
            return True
        else:
            print(f"❌ GDELT returned status: {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ GDELT error: {e}")
        return False

def get_domain(url):
    """Get domain from URL"""
    try:
        return urlparse(url).netloc.lower().replace('www.', '')
    except:
        return ""

def format_date(dt):
    """Format datetime for GDELT"""
    return dt.strftime("%Y%m%d%H%M%S")

def build_query(company, ticker):
    """Build search query"""
    return f'("{company}" OR {ticker}) AND sourcelang:english'

def get_articles(ticker, company, start_date, end_date):
    """Get articles for one asset and time period"""
    
    query = build_query(company, ticker)
    
    params = {
        "query": query,
        "mode": "ArtList", 
        "format": "json",
        "startdatetime": format_date(start_date),
        "enddatetime": format_date(end_date),
        "maxrecords": str(MAX_ARTICLES),
        "sort": "DateAsc"
    }
    
    print(f"   📡 Requesting {ticker} articles...")
    
    try:
        response = requests.get(API_URL, params=params, timeout=30)
        
        if response.status_code == 200:
            data = response.json() or {}
            articles = data.get("articles", [])
            print(f"   ✅ Got {len(articles)} articles for {ticker}")
            return articles
        else:
            print(f"   ❌ Error {response.status_code} for {ticker}")
            return []
            
    except Exception as e:
        print(f"   ❌ Request failed for {ticker}: {e}")
        return []

def process_articles(articles, ticker, company):
    """Convert raw articles to clean format"""
    
    processed = []
    
    for article in articles:
        url = article.get("url", "")
        if not url:
            continue
            
        domain = article.get("domain") or get_domain(url)
        seen_date = article.get("seendate", "")
        
        # Parse date
        if seen_date and len(seen_date) >= 14:
            try:
                clean_date = seen_date.replace('T', '').replace('Z', '')
                parsed_date = f"{clean_date[:4]}-{clean_date[4:6]}-{clean_date[6:8]} {clean_date[8:10]}:{clean_date[10:12]}:{clean_date[12:14]}"
            except:
                parsed_date = "2024-01-01 00:00:00"
        else:
            parsed_date = "2024-01-01 00:00:00"
        
        processed.append({
            "ticker": ticker,
            "company": company,
            "date": parsed_date,
            "title": article.get("title", ""),
            "url": url,
            "domain": domain,
            "is_quality": domain in QUALITY_SOURCES,
            "source": "gdelt"
        })
    
    return processed

def collect_recent_news(months_back=12):
    """Collect recent news (simple approach)"""
    
    print(f"\n📰 Collecting last {months_back} months of news...")
    
    # Calculate date range
    end_date = datetime.now()
    start_date = datetime(end_date.year, end_date.month - months_back, 1)
    
    print(f"📅 Period: {start_date.date()} to {end_date.date()}")
    
    all_articles = []
    stats = {}
    
    for i, (ticker, company) in enumerate(ASSETS.items(), 1):
        print(f"\n📊 Processing {i}/{len(ASSETS)}: {ticker} - {company}")
        
        # Get articles
        raw_articles = get_articles(ticker, company, start_date, end_date)
        
        # Process articles
        processed = process_articles(raw_articles, ticker, company)
        all_articles.extend(processed)
        
        # Stats
        quality_count = sum(1 for a in processed if a['is_quality'])
        stats[ticker] = {
            'total': len(processed),
            'quality': quality_count,
            'quality_rate': quality_count / len(processed) if processed else 0
        }
        
        print(f"   📰 {len(processed)} articles ({quality_count} quality)")
        
        # Rate limiting (important!)
        if i < len(ASSETS):
            print(f"   ⏱️ Waiting {SLEEP_TIME} seconds...")
            time.sleep(SLEEP_TIME)
    
    return all_articles, stats

def collect_full_period(start_year=2015, end_year=2024):
    """Collect full historical period (year by year)"""
    
    print(f"\n📰 Collecting full period: {start_year}-{end_year}")
    print("⚠️ This will take a while! (~30 minutes)")
    
    all_articles = []
    total_stats = {}
    
    for year in range(start_year, end_year + 1):
        print(f"\n📅 Collecting year {year}...")
        
        start_date = datetime(year, 1, 1)
        end_date = datetime(year, 12, 31)
        
        year_articles = []
        
        for i, (ticker, company) in enumerate(ASSETS.items(), 1):
            print(f"   📊 {ticker} ({i}/{len(ASSETS)}) for {year}")
            
            # Get articles for this year
            raw_articles = get_articles(ticker, company, start_date, end_date)
            processed = process_articles(raw_articles, ticker, company)
            year_articles.extend(processed)
            
            # Update stats
            if ticker not in total_stats:
                total_stats[ticker] = {'total': 0, 'quality': 0}
            
            quality_count = sum(1 for a in processed if a['is_quality'])
            total_stats[ticker]['total'] += len(processed)
            total_stats[ticker]['quality'] += quality_count
            
            print(f"      ✅ {len(processed)} articles ({quality_count} quality)")
            
            # Rate limiting
            time.sleep(SLEEP_TIME)
        
        all_articles.extend(year_articles)
        print(f"   📊 Year {year} total: {len(year_articles)} articles")
    
    return all_articles, total_stats

def save_results(articles, stats, filename="gdelt_news_data.csv"):
    """Save results to files"""
    
    if not articles:
        print("❌ No articles to save")
        return
    
    # Create DataFrame
    df = pd.DataFrame(articles)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.sort_values(['ticker', 'date']).reset_index(drop=True)
    
    # Remove duplicates
    initial_count = len(df)
    df = df.drop_duplicates(subset=['url'], keep='first')
    final_count = len(df)
    
    print(f"\n🧹 Removed {initial_count - final_count} duplicates")
    
    # Save main file
    output_path = f"data/raw/news/{filename}"
    df.to_csv(output_path, index=False)
    
    print(f"💾 Saved: {output_path}")
    print(f"📊 Final dataset: {len(df)} articles")
    print(f"📅 Date range: {df['date'].min()} to {df['date'].max()}")
    
    # Quality analysis
    quality_articles = df[df['is_quality'] == True]
    print(f"🏆 Quality articles: {len(quality_articles)}/{len(df)} ({len(quality_articles)/len(df):.1%})")
    
    # Save stats
    stats_with_summary = {
        'collection_stats': stats,
        'summary': {
            'total_articles': len(df),
            'quality_articles': len(quality_articles),
            'quality_rate': len(quality_articles)/len(df),
            'date_range': {
                'start': str(df['date'].min()),
                'end': str(df['date'].max())
            },
            'assets_covered': len(df['ticker'].unique()),
            'collection_date': datetime.now().isoformat()
        }
    }
    
    stats_path = f"data/raw/news/{filename.replace('.csv', '_stats.json')}"
    with open(stats_path, 'w') as f:
        json.dump(stats_with_summary, f, indent=2, default=str)
    
    print(f"📈 Stats saved: {stats_path}")
    
    # Show breakdown by asset
    print(f"\n📋 Articles by asset:")
    for ticker in sorted(df['ticker'].unique()):
        ticker_df = df[df['ticker'] == ticker]
        quality_count = len(ticker_df[ticker_df['is_quality'] == True])
        print(f"   {ticker}: {len(ticker_df)} total ({quality_count} quality)")
    
    return df

# ============================================================================
# MAIN FUNCTIONS
# ============================================================================

def quick_test():
    """Quick test with just 2 assets"""
    
    print("🧪 QUICK TEST - 2 assets, recent data")
    
    # Test GDELT first
    if not test_gdelt():
        return None
    
    # Use only 2 assets for test
    test_assets = dict(list(ASSETS.items())[:2])
    
    print(f"📊 Testing with: {list(test_assets.keys())}")
    
    # Get recent articles (last 3 months)
    end_date = datetime.now()
    start_date = datetime(end_date.year, end_date.month - 3, 1)
    
    all_articles = []
    
    for ticker, company in test_assets.items():
        print(f"\n📰 Testing {ticker}...")
        
        raw_articles = get_articles(ticker, company, start_date, end_date)
        processed = process_articles(raw_articles, ticker, company)
        all_articles.extend(processed)
        
        time.sleep(SLEEP_TIME)
    
    # Save test results
    if all_articles:
        df = save_results(all_articles, {}, "test_gdelt_data.csv")
        print("✅ Quick test successful!")
        return df
    else:
        print("❌ Quick test failed - no articles collected")
        return None

def collect_recent():
    """Collect recent news (last 12 months)"""
    
    print("📰 COLLECTING RECENT NEWS (12 months)")
    
    # Test first
    if not test_gdelt():
        return None
    
    # Collect recent news
    articles, stats = collect_recent_news(months_back=12)
    
    # Save results
    if articles:
        df = save_results(articles, stats, "recent_gdelt_news.csv")
        return df
    else:
        print("❌ No articles collected")
        return None

def collect_full():
    """Collect full historical data"""
    
    print("📰 COLLECTING FULL HISTORICAL DATA (2015-2024)")
    print("⚠️ This will take ~30-60 minutes!")
    
    # Test first
    if not test_gdelt():
        return None
    
    # Collect full period
    articles, stats = collect_full_period(start_year=2015, end_year=2024)
    
    # Save results
    if articles:
        df = save_results(articles, stats, "full_gdelt_news.csv")
        return df
    else:
        print("❌ No articles collected")
        return None

# ============================================================================
# SIMPLE EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("🎯 SIMPLE GDELT SCRAPER OPTIONS:")
    print("1. quick_test() - Test with 2 assets")
    print("2. collect_recent() - Last 12 months")  
    print("3. collect_full() - Full 2015-2024 data")
    print("\n💡 Start with: quick_test()")
    
    # Uncomment to run:
    # quick_test()
    # collect_recent()
    collect_full()



🚀 Simple GDELT News Scraper
📊 Assets: 17
⏱️ Sleep time: 5 seconds
🎯 SIMPLE GDELT SCRAPER OPTIONS:
1. quick_test() - Test with 2 assets
2. collect_recent() - Last 12 months
3. collect_full() - Full 2015-2024 data

💡 Start with: quick_test()
📰 COLLECTING FULL HISTORICAL DATA (2015-2024)
⚠️ This will take ~30-60 minutes!
🧪 Testing GDELT...
✅ GDELT working! Got 5 test articles

📰 Collecting full period: 2015-2024
⚠️ This will take a while! (~30 minutes)

📅 Collecting year 2015...
   📊 NVDA (1/17) for 2015
   📡 Requesting NVDA articles...
   ❌ Request failed for NVDA: Expecting value: line 1 column 1 (char 0)
      ✅ 0 articles (0 quality)
   📊 MSFT (2/17) for 2015
   📡 Requesting MSFT articles...
   ❌ Request failed for MSFT: Expecting value: line 1 column 1 (char 0)
      ✅ 0 articles (0 quality)
   📊 GOOGL (3/17) for 2015
   📡 Requesting GOOGL articles...
   ❌ Request failed for GOOGL: Expecting value: line 1 column 1 (char 0)
      ✅ 0 articles (0 quality)
   📊 AMD (4/17) for 2015
   📡 

KeyboardInterrupt: 

In [2]:
collect_full()

📰 COLLECTING FULL HISTORICAL DATA (2015-2024)
⚠️ This will take ~30-60 minutes!
🧪 Testing GDELT...
✅ GDELT working! Got 5 test articles

📰 Collecting full period: 2015-2024
⚠️ This will take a while! (~30 minutes)

📅 Collecting year 2015...
   📊 NVDA (1/17) for 2015
   📡 Requesting NVDA articles...
   ❌ Request failed for NVDA: Expecting value: line 1 column 1 (char 0)
      ✅ 0 articles (0 quality)
   📊 MSFT (2/17) for 2015
   📡 Requesting MSFT articles...
   ❌ Request failed for MSFT: Expecting value: line 1 column 1 (char 0)
      ✅ 0 articles (0 quality)
   📊 GOOGL (3/17) for 2015
   📡 Requesting GOOGL articles...
   ❌ Request failed for GOOGL: Expecting value: line 1 column 1 (char 0)
      ✅ 0 articles (0 quality)
   📊 AMD (4/17) for 2015
   📡 Requesting AMD articles...
   ❌ Request failed for AMD: Expecting value: line 1 column 1 (char 0)
      ✅ 0 articles (0 quality)
   📊 MU (5/17) for 2015
   📡 Requesting MU articles...
   ❌ Request failed for MU: Expecting value: line 1 colu

KeyboardInterrupt: 

In [ ]:
# ============================================================================
# USAGE EXAMPLES
# ============================================================================

"""
# OPTION 1: Quick test first
result = quick_test()

# OPTION 2: If test works, collect recent data
result = collect_recent()

# OPTION 3: Full historical collection (long running)
result = collect_full()

# Check your results
if result is not None:
    print(f"Success! Collected {len(result)} articles")
    print(f"Files saved in: data/raw/news/")
"""